# Public archival edition
Original academic source with outputs and execution metadata removed.
Licensed input data are excluded. See ../docs/EVIDENCE_AND_LIMITATIONS.md before interpreting calculations.
This notebook has not been independently rerun or certified as a valid trading backtest.

# Black_Litterman PORTFOLIO CONSTRUCTION

In [ ]:
import pandas as pd
import numpy as np
import scipy.optimize as sc_optim
import imageio
import matplotlib
import matplotlib.pyplot as plt
import time

## Step1. Underlying data assignment

In [ ]:
# original data
## Stock price data
PRICE_FILENAME = './Data/Price_Data.xlsx'
PRICE_SHEETNAME= 'Weekly'

In [ ]:
## Stock market capitalization data
MV_FILENAME = './Data/Market_Value.xlsx'
MV_SHEETNAME= 'Weekly'

In [ ]:
# model parameter
TAU = 0.3           # The scaling of the posterior expected return covariance matrix ranges from 0.004 to 0.4

In [ ]:
# Model back test
## Back test parameters
BACK_TEST_T = 200   # Backtest time T window: 200 periods
START_INDEX = 273   # Start date: 2015/1/2
END_INDEX = 324     # End date: 2015/12/25
INDEX_NUMBER = 0    # Stock Index data index: 0. S&p 500,1. Dow Jones, 2. Nasdaq

In [ ]:
## Parameters of drawing
BACK_TEST_X_LABEL = 'Week'
BACK_TEST_Y_LABEL = 'Accumulated Return(log)'
BACK_TEST_PERIOD_NAME = '2015'

In [ ]:
# Point of view parameter
VIEW_TYPE = 2       # Index the list of opinions
VIEW_TYPE_NAME = ['Market value as view', "Arbitrary views", "Reasonable views", "Near period return as view"]
VIEW_T = 10         # When the view is "Near period return as view", the near-term parameter needs to be defined, that is, the VIEW_T period historical return is averaged as the expected return

In [ ]:
# Investment category
## Stock parameters
stock_cc_ret = 0
stock_names = 0
stock_number = 0
market_value_weight = 0
## Stock index parameters
index_num = INDEX_NUMBER
index_name = 0
index_cc_ret = 0

## Step2. Print important parameters

In [ ]:
def print_data(index_cc_ret,stock_cc_ret,index_name,stock_names):
    print(index_cc_ret)                # Stock index (S&P 500) yield
    print(stock_cc_ret)                # The yield of 10 stocks
    print(index_name)                  # Stock index name: S&P 500
    print(stock_names)                 # List of stock names

## Step3. Read data

In [ ]:
def read_data(filename, sheet_name):  # Read the data in excel sheet
    df = pd.read_excel(filename, sheet_name = sheet_name)
    df.set_index("Date", inplace=True)      # Indexed by the Date column
    df.index = range(len(df))               # Turn the index into a number
    df = df.astype('float64')               # Convert the data to float64 format
    return df

In [ ]:
read_data(PRICE_FILENAME,PRICE_SHEETNAME)

In [ ]:
read_data(MV_FILENAME,MV_SHEETNAME)

## Step4. The initial stock data is processed to obtain the list of stock names, rate of return and other data

In [ ]:
def get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME):
    st=read_data(PRICE_FILENAME,PRICE_SHEETNAME)

    # shift(): Move one unit downward
    #It is equivalent to dividing each cell by the previous cell to obtain the rate of return, and then taking the logarithmic form, which is the approximate form of (P2-P1)/P1
    #Since the first row has no divisible data, it is deleted, and the final data is 525-1=524
    log_ret = np.log(st/st.shift())
    log_ret = log_ret.drop(index=[0])

    # Convert three index, ten stock names into a list of names
    names = log_ret.columns.tolist()

    # index name: S&P 500
    index_name = names[index_num]

    # Stock name: 3rd and subsequent data
    stock_names = names[3:]

    # index_cc_ret：S&p 500 yield
    index_cc_ret = log_ret[index_name]

    # stock_cc_ret：Table of returns for 10 stocks
    stock_cc_ret = log_ret[stock_names]

    # Update assignment to the data
    stock_number = len(stock_names)

    return log_ret,index_cc_ret,stock_cc_ret

In [ ]:
get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)

## Step5. The initial market value data are processed to obtain the market_value_weight matrix

In [ ]:
def get_market_value_weight(MV_FILENAME,MV_SHEETNAME):
    mv=read_data(MV_FILENAME,MV_SHEETNAME)

    # Since the last column contains the item "Total", we need to slice [0:-1]
    stock_names= mv.columns.tolist()[0:-1]

    # calculate market value weight
    for n in stock_names:
        mv[n] = mv[n] / mv["TOTAL"]

    # Drop the first column to keep the quantity consistent with the yield file
    mv = mv.drop(index=[0])

    # The column "Total" is removed and only 10 stock market capitalization weight data are retained
    mv = mv[stock_names]

    # The market capitalization weights are saved in matrix form
    market_value_weight = np.array(mv)

    return  mv,market_value_weight

In [ ]:
get_market_value_weight(MV_FILENAME,MV_SHEETNAME)

## Step6. Calculate the prior expected return: implied_ret

In [ ]:
def get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd):

    # weekly risk-free cc return = ln(1+3.24%)/(365/7) = 0.0006132
    rf = 0.0006132

    # The covariance matrix is calculated from the stock returns：mkt_cov
    mkt_cov = np.array(stock_cc_ret.cov())

    # Calculate prior expected return：implied_ret
    implied_ret = lambd * np.dot(mkt_cov, w_mkt)
    return implied_ret,mkt_cov

## Step7. Set the opinion matrix P and the relative return vector Q

In [ ]:
def get_views_P_Q_matrix(view_type, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T):
    st = read_data(PRICE_FILENAME, PRICE_SHEETNAME)
    log_ret = np.log(st/st.shift())
    log_ret = log_ret.drop(index=[0])
    # Convert three index, ten stock names into a list of names
    names = log_ret.columns.tolist()

    # Stock name: 3rd and subsequent data
    stock_names = names[3:]

    # Update assignment to the data
    stock_number = len(stock_names)
    N = stock_number

    if (view_type == 0 or view_type == 1):
        # view_type = 0: The investor has no opinion and uses the current market value weight (that is, the weight in equilibrium) as the weight of the portfolio
        # view_type = 1: Assign investors arbitrary opinions, here 3 opinions are randomly assigned
        '''
        Opinion 1. Berkshire Hathaway beats Exxon's expected earnings by 0.01%;
        Opinion 2. Microsoft beat jpmorgan's expected earnings by 0.025%;
        Opinion 3. A 10% Morgan +90%VISA portfolio is expected to return 0.01% more than a 10% Walmart +90% Bank of America portfolio
        '''
        P = np.zeros([3, N])
        P[0, 8] = 1
        P[0, 9] = -1
        P[1, 1] = 1
        P[1, 3] = -1
        P[2, 3] = 0.1
        P[2, 4] = 0.9
        P[2, 6] = -0.1
        P[2, 7] = -0.9
        Q = np.array([0.0001, 0.00025, 0.0001])
    
    elif (view_type == 2):
        # view_type = 2: Reasonable views
        P = np.zeros([1, N])
        P[0, 2] = 1
        P[0, 3] = -1
        Q = [0.017]
    
    elif (view_type == 3):
        # view_type = 3: The historical average rate of return in the most recent VIEW_T period is selected as the expected rate of return
        # T_near: The historical average return of the recent T_near period data is used as the expected return
        T_near = VIEW_T
        P = np.identity(N)
        stock_cc_ret_near = stock_cc_ret.iloc[-T_near:]
        Q = np.array(stock_cc_ret_near.mean())
    
    else:
        print("There is no such kind of view type!")

    return P, Q


In [ ]:

view_type=1

a,b=get_views_P_Q_matrix(view_type, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
print(a,b)

## Step8. Calculate the Omega matrix

In [ ]:
def get_views_omega( mkt_cov, P,TAU):
    # K: The number of investor views
    K = len(P)
    # Generate a diagonal matrix of K dimensions (all ones on the diagonal)
    omega = np.identity(K)
    for i in range(K):
        # Select P row by row (Views matrix, dimension: K*N, N=10 here)
        P_i = P[i]
        omg_i = np.dot(np.dot(P_i, mkt_cov), P_i.T) *TAU
        # Assign the resulting result to the diagonal elements of the matrix
        omega[i][i] = omg_i
    return omega,K

In [ ]:
st=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
log_ret = np.log(st/st.shift())
log_ret = log_ret.drop(index=[0])

names = log_ret.columns.tolist()
stock_names = names[3:]
stock_cc_ret = log_ret[stock_names]
mkt_cov = np.array(stock_cc_ret.cov())
P=a
get_views_omega( mkt_cov, P,TAU)

## Step9. Calculate posterior expected return :mu_p

In [ ]:
def get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega):
    # tau is the scaling scale
    # From posterior expected return, the formula for calculating mu_p
    k = np.linalg.inv(np.linalg.inv(TAU * mkt_cov) + np.dot(np.dot(P.T, np.linalg.inv(omega)), P))
    posterior_ret = np.dot(k, np.dot(np.linalg.inv(TAU * mkt_cov), implied_ret)+np.dot(np.dot(P.T, np.linalg.inv(omega)), Q))
    return posterior_ret

## Step10. Calculate the new weight, weight_bl, obtained from the bl model

In [ ]:

def get_weight_bl( posterior_ret, mkt_cov, lambd):
    weight_bl = np.dot(np.linalg.inv(lambd * mkt_cov), posterior_ret)
    return weight_bl

In [ ]:
# Visualization


def show(stock_names,weights):
    # Creating the bar plot
    plt.figure(figsize=(10, 6))
    plt.bar(stock_names, weights, color='skyblue')
    plt.xlabel('Stock Names')
    plt.ylabel('Weights')
    plt.title('Visualization of Weight Distribution Across Different Stocks')
    plt.show()

st=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
names = st.columns.tolist()
stock_names = names[3:]

# OPTIMAZATION METHOD :MEAN VARIANCE

## assume $\lambda =4.5$ (namely Trustee Investor)

In [ ]:
lambd=4.5
for i in range(1,4):
    VIEW_TYPE=i
    
    df_P=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
    
    df_M=read_data(MV_FILENAME,MV_SHEETNAME)
    
    log_ret,index_cc_ret,stock_cc_ret=get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)
    
    mv,market_value_weight=get_market_value_weight(MV_FILENAME,MV_SHEETNAME)
    
    w_mkt = market_value_weight[-1, :]  # Select the last line of the DataFrame
    
    implied_ret,mkt_cov=get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd)
    
    P, Q=get_views_P_Q_matrix(VIEW_TYPE, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
    
    omega,K=get_views_omega(mkt_cov, P,TAU)
    
    posterior_ret=get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega)
    
    weight_bl=get_weight_bl(posterior_ret, mkt_cov, lambd)
    
    print(weight_bl)
    show(stock_names,weight_bl)

## assume $\lambda =2.5$      (namely Market Investor)

In [ ]:
lambd=2.5
for i in range(1,4):
    VIEW_TYPE=i
    
    df_P=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
    
    df_M=read_data(MV_FILENAME,MV_SHEETNAME)
    
    log_ret,index_cc_ret,stock_cc_ret=get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)
    
    mv,market_value_weight=get_market_value_weight(MV_FILENAME,MV_SHEETNAME)
    
    w_mkt = market_value_weight[-1, :]  # Select the last line of the DataFrame
    
    implied_ret,mkt_cov=get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd)
    
    P, Q=get_views_P_Q_matrix(VIEW_TYPE, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
    
    omega,K=get_views_omega(mkt_cov, P,TAU)
    
    posterior_ret=get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega)
    
    weight_bl=get_weight_bl(posterior_ret, mkt_cov, lambd)
    
    print(weight_bl)
    show(stock_names,weight_bl)

## assume $\lambda =0.001$ (namely Kelly Investor)

In [ ]:
lambd=0.001
for i in range(1,4):
    VIEW_TYPE=i
    
    df_P=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
    
    df_M=read_data(MV_FILENAME,MV_SHEETNAME)
    
    log_ret,index_cc_ret,stock_cc_ret=get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)
    
    mv,market_value_weight=get_market_value_weight(MV_FILENAME,MV_SHEETNAME)
    
    w_mkt = market_value_weight[-1, :]  # Select the last line of the DataFrame
    
    implied_ret,mkt_cov=get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd)
    
    P, Q=get_views_P_Q_matrix(VIEW_TYPE, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
    
    omega,K=get_views_omega(mkt_cov, P,TAU)
    
    posterior_ret=get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega)
    
    weight_bl=get_weight_bl(posterior_ret, mkt_cov, lambd)
    
    print(weight_bl)
    show(stock_names,weight_bl)

# OPTIMAZATION METHOD :SHARP RATIO

$w^*=\frac{\sum^{-1}(\mu-r_f 1}{1^t\sum^{-1}(\mu-r_f1)}$

In [ ]:
def Max_SR(mu,rf,Sigma):
    # Calculate excess return
    excess_returns = mu - rf

    # Calculate the weight
    Sigma_inv = np.linalg.inv(Sigma)  # Inverse of the covariance matrix
    w_star_numerator = np.dot(Sigma_inv, excess_returns)
    w_star_denominator = np.dot(np.ones(mu.shape), w_star_numerator)
    w_star = w_star_numerator / w_star_denominator

    print("Optimal Portfolio Weights: ", w_star)
    return w_star

## assume $\lambda =4.5$ (namely Trustee Investor)

In [ ]:
lambd=4.5
for i in range(1,4):
    VIEW_TYPE=i
    
    df_P=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
    
    df_M=read_data(MV_FILENAME,MV_SHEETNAME)
    
    log_ret,index_cc_ret,stock_cc_ret=get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)
    
    mv,market_value_weight=get_market_value_weight(MV_FILENAME,MV_SHEETNAME)
    
    w_mkt = market_value_weight[-1, :]  # Select the last line of the DataFrame
    
    implied_ret,mkt_cov=get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd)
    
    P, Q=get_views_P_Q_matrix(VIEW_TYPE, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
    
    omega,K=get_views_omega(mkt_cov, P,TAU)
    
    posterior_ret=get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega)
    
    rf=1.5600747436120286/13    #The average yield on the three-month Treasury bill: 1.5600747436120286%
    
    weight_bl=Max_SR(posterior_ret,rf,mkt_cov)
    show(stock_names,weight_bl)

## assume $\lambda =2.5$ (namely Market Investor)

In [ ]:
lambd=2.5
for i in range(1,4):
    VIEW_TYPE=i
    
    df_P=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
    
    df_M=read_data(MV_FILENAME,MV_SHEETNAME)
    
    log_ret,index_cc_ret,stock_cc_ret=get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)
    
    mv,market_value_weight=get_market_value_weight(MV_FILENAME,MV_SHEETNAME)
    
    w_mkt = market_value_weight[-1, :]  # Select the last line of the DataFrame
    
    implied_ret,mkt_cov=get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd)
    
    P, Q=get_views_P_Q_matrix(VIEW_TYPE, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
    
    omega,K=get_views_omega(mkt_cov, P,TAU)
    
    posterior_ret=get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega)
    
    rf=1.5600747436120286/13    #The average yield on the three-month Treasury bill: 1.5600747436120286%
    
    weight_bl=Max_SR(posterior_ret,rf,mkt_cov)
    show(stock_names,weight_bl)

## assume $\lambda =0.001$ (namely Kelly Investor)

In [ ]:
lambd=0.001
for i in range(1,4):
    VIEW_TYPE=i
    
    df_P=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
    
    df_M=read_data(MV_FILENAME,MV_SHEETNAME)
    
    log_ret,index_cc_ret,stock_cc_ret=get_cc_return(PRICE_FILENAME,PRICE_SHEETNAME)
    
    mv,market_value_weight=get_market_value_weight(MV_FILENAME,MV_SHEETNAME)
    
    w_mkt = market_value_weight[-1, :]  # Select the last line of the DataFrame
    
    implied_ret,mkt_cov=get_implied_excess_equilibrium_return(stock_cc_ret, w_mkt,lambd)
    
    P, Q=get_views_P_Q_matrix(VIEW_TYPE, stock_cc_ret, PRICE_FILENAME, PRICE_SHEETNAME, VIEW_T)
    
    omega,K=get_views_omega(mkt_cov, P,TAU)
    
    posterior_ret=get_posterior_combined_return(TAU,implied_ret, mkt_cov, P, Q, omega)
    
    rf=1.5600747436120286/13    #The average yield on the three-month Treasury bill: 1.5600747436120286%
    
    weight_bl=Max_SR(posterior_ret,rf,mkt_cov)
    show(stock_names,weight_bl)